# Fit a SurfaceShearJAX to the NCSX plasma surface

Objective: integrated squared facet distance from the free shear surface to a triangulated mesh of the target (`surface_distance.surface_surface_distance`).

Optimizer: L-BFGS-B with box bounds from `SurfaceShearJAX.get_bounds()`. Jacobian via JAX autodiff. History logged every 20 iterations.

In [ ]:
import time
import matplotlib.pyplot as plt
import jax.numpy as jnp
import numpy as np
from jax import grad, jacfwd, jit
from scipy.optimize import minimize
from simsopt import load

from quadcoil.surface import SurfaceJAX
from surface_distance import surface_surface_distance
from surface_transport import SurfaceShearJAX, SurfaceBeltramiJAX
from utils import plot_gamma, to_vtk

def plot_gamma(surf, close=True, ax=None, **kwargs):
    """Plot ``surf.gamma()`` of a ``SurfaceJAX`` as a 3D surface, equal aspect."""
    gamma = surf.gamma()
    if close:
        # quadpoints are open grids in [0, 1); wrap once in theta (and in phi
        # only if the phi grid spans the whole torus) so the mesh has no seam.
        gamma = np.concatenate([gamma, gamma[:, :1]], axis=1)
        if np.isclose(float(surf.quadpoints_phi[-1] + surf.dphi), 1.0):
            gamma = np.concatenate([gamma, gamma[:1]], axis=0)

    if ax is None:
        ax = plt.figure(figsize=(7, 6)).add_subplot(projection='3d')
    x, y, z = gamma[..., 0], gamma[..., 1], gamma[..., 2]
    ax.plot_surface(
        x, y, z,
        **{'rstride': 1, 'cstride': 1, 'linewidth': 0,
           'antialiased': False, 'cmap': 'viridis', **kwargs},
    )
    ax.set_aspect('equal')  # matplotlib >= 3.6
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    return ax

In [ ]:
# Target: second surface in ncsx.json (plasma-scale NCSX)
_, simsopt_surface = load('ncsx.json')
target_surface = SurfaceJAX.from_simsopt(simsopt_surface)
target_volume = simsopt_surface.volume()
minor_radius_target = np.sqrt(target_volume / (2 * np.pi**2) / simsopt_surface.major_radius())
print('Target surface volume:    ', target_volume)
print('Seed surface minor radius:', minor_radius_target)
# Facet-mesh resolution for the distance objective
n_phi, n_theta = 16, 16 # 32, 32

In [ ]:

# Initial shear surface: circular seed from major/minor radius, zero dofs
init_surface_b = SurfaceBeltramiJAX(
    nfp=simsopt_surface.nfp,
    stellsym=simsopt_surface.stellsym,
    seed_major_radius=simsopt_surface.major_radius(),
    seed_minor_radius=minor_radius_target,
    # Half of simsopt's 32x32 grid: jacfwd needs 5.2 GB at 32x32 vs 1.8 GB
    # here, and the GPU has 8 GB. phi spans one field period, theta the full
    # poloidal range, matching simsopt's convention.
    quadpoints_phi=jnp.linspace(0.0, 1.0 / simsopt_surface.nfp, 16, endpoint=False),
    quadpoints_theta=jnp.linspace(0.0, 1.0, 16, endpoint=False),
    step_num=10, # Step number for time evolution
    m_per_period=3,
    min_lam=1e-5,
    max_lam=7,
    n_lam=1000,
    # dofs=None,
    # box_r=None, # By default based on the major and minor radius
    # box_z=None,
    max_order=12, # 50, # Order of the infinite series that make up of the analytic sln
    max_iter=100,
    tol=1e-5,
    pi_guard=1e-3,
    polish=True,
)

# # Facet-mesh resolution for the distance objective
# n_phi, n_theta = 32, 32

In [ ]:
print(init_surface_b.basis.m)
print(init_surface_b.basis.lam)

In [ ]:
def make_surface_b(dofs):
    """Copy of init_surface with the given dofs."""
    return SurfaceBeltramiJAX(
        nfp=init_surface_b.nfp,
        stellsym=init_surface_b.stellsym,
        seed_major_radius=init_surface_b.seed_major_radius,
        seed_minor_radius=init_surface_b.seed_minor_radius,
        quadpoints_phi=init_surface_b.quadpoints_phi,
        quadpoints_theta=init_surface_b.quadpoints_theta,
        step_num=init_surface_b.step_num,
        m_per_period=init_surface_b.m_per_period,
        min_lam=init_surface_b.min_lam,
        max_lam=init_surface_b.max_lam,
        n_lam=init_surface_b.n_lam,
        dofs=dofs,
        box_r=init_surface_b.box_r,
        box_z=init_surface_b.box_z,
        max_order=init_surface_b.max_order,
        max_iter=init_surface_b.max_iter,
        tol=init_surface_b.tol,
        pi_guard=init_surface_b.pi_guard,
        polish=init_surface_b.polish,
    )

@jit
def f_b(dofs):
    temp_surface = make_surface_b(dofs)
    return surface_surface_distance(
        target_surface, temp_surface, n_phi, n_theta
    )


# Forward mode, not grad: reverse-mode AD through the RK4 Beltrami flow needs
# ~27 GB and is OOM-killed while XLA compiles. With only len(basis) dofs
# forward mode is also the cheaper choice.
df_b = jit(jacfwd(f_b))

def f_b_np(x):
    return float(f_b(jnp.asarray(x)))
    
def jac_b_np(x):
    return np.asarray(df_b(jnp.asarray(x)), dtype=float)

In [ ]:
dofs_test = np.zeros(len(init_surface_b.basis.m))
dofs_test[6] = 0.1
dofs_test[7] = 0.2

In [ ]:
surf_test = make_surface_b(dofs_test)
# to_vtk(surf_test, 'test_beltrami_surface')

plot_gamma(surf_test)

In [ ]:
# History buffers (filled every L-BFGS-B iteration)
n_iter_b_history, dofs_b_history, f_b_history, time_b_history = [], [], [], []
state = {'n_iter': 0}


def callback_b(intermediate_result):
    # Naming the argument ``intermediate_result`` opts into scipy's OptimizeResult
    # callback, so .fun is already computed instead of costing another ~10 s
    # objective evaluation. Logging every iteration, not every 20th: this problem
    # converges in well under 20 iterations.
    state['n_iter'] += 1
    print('f', intermediate_result.fun)
    n_iter_b_history.append(state['n_iter'])
    dofs_b_history.append(np.asarray(intermediate_result.x).copy())
    f_b_history.append(float(intermediate_result.fun))
    time_b_history.append(time.time())


# lb, ub = init_surface_b.get_bounds()
# bounds = list(zip(np.asarray(lb), np.asarray(ub)))

result = minimize(
    f_b_np,
    x0=np.asarray(init_surface_b.dofs, dtype=float),
    jac=jac_b_np,
    method='L-BFGS-B',  # box-constrained
    # bounds=bounds,
    callback=callback_b,
    options={'maxiter': 200},  # ~40 s per iteration
)

print(result.message)
print('nit =', result.nit, '  fun =', result.fun)
print('logged iterations:', n_iter_b_history)

In [ ]:
plt.plot(np.array(time_b_history)-time_b_history[0], f_b_history)